In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
key = os.getenv("OPENAI_API_KEY")

In [3]:
client = OpenAI(
    api_key = os.getenv("OPENAI_API_KEY"),
    base_url = os.getenv("OPENAI_BASE_URL")
)


In [4]:
def multiply (a :int, b: int) -> int:
    return a * b
multiply (747,737)

550539

In [19]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "multiplies two numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "first number to multiply"
                    },
                    "b": {
                        "type": "number",
                        "description": "second number to multiply"
                    }
                },
                "required": ["a", "b"]
            }
        }
    }
]
]


SyntaxError: unmatched ']' (2703112576.py, line 24)

In [1]:
import json
import psutil
import time
from openai import OpenAI

client = OpenAI() # کلید API و تنظیمات اولیه خودت

# ۱. تعریف توابع
def get_process_stats(process_name: str):
    total_memory = 0
    total_cpu = 0
    found = False
    for proc in psutil.process_iter(['name', 'memory_info', 'cpu_percent']):
        try:
            if process_name.lower() in proc.info['name'].lower():
                found = True
                total_memory += proc.info['memory_info'].rss / (1024 * 1024)
                total_cpu += proc.info['cpu_percent']
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    if found:
        return f"برنامه {process_name}: مصرف رم {total_memory:.2f} MB - مصرف پردازنده {total_cpu}%"
    return f"برنامه‌ای با نام {process_name} پیدا نشد."

def get_disk_space(path: str = "/"):
    try:
        usage = psutil.disk_usage(path)
        total_gb = usage.total / (1024**3)
        free_gb = usage.free / (1024**3)
        percent = usage.percent
        return f"مسیر {path}: کل فضا {total_gb:.2f}GB | فضای آزاد {free_gb:.2f}GB | درصد مصرف {percent}%"
    except Exception as e:
        return f"خطا در بررسی مسیر {path}: {str(e)}"

def get_system_uptime():
    boot_time = psutil.boot_time()
    uptime_seconds = time.time() - boot_time
    hours = int(uptime_seconds // 3600)
    minutes = int((uptime_seconds % 3600) // 60)
    return f"سیستم به مدت {hours} ساعت و {minutes} دقیقه است که روشن است."

# ۲. تعریف اسکیماها
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_process_stats",
            "description": "استخراج میزان مصرف رم و پردازنده یک برنامه مشخص در اوبونتو",
            "parameters": {
                "type": "object",
                "properties": {
                    "process_name": {"type": "string", "description": "نام برنامه مثلا firefox یا python3"}
                },
                "required": ["process_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_disk_space",
            "description": "بررسی میزان فضای کل و آزاد دیسک برای یک مسیر مشخص",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "مسیر مورد نظر در سیستم‌عامل مثل / یا /home"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_system_uptime",
            "description": "دریافت مدت زمانی که سیستم‌عامل اوبونتو روشن بوده است",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]

available_functions = {
    "get_process_stats": get_process_stats,
    "get_disk_space": get_disk_space,
    "get_system_uptime": get_system_uptime
}

# ۳. تابع اصلی ایجنت با چاپ جزییات Responseها
def run_agent(user_query: str):
    print(f"\n==================== سوال کاربر: '{user_query}' ====================")
    messages = [{"role": "user", "content": user_query}]
    
    # درخواست اول
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    print("\n--- [Response اول از مدل (تصمیم‌گیری)] ---")
    print(response_message)
    
    if response_message.tool_calls:
        messages.append(response_message)
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"\n--- [اجرای تابع پایتون: {function_name} با ورودی {function_args}] ---")
            
            function_to_call = available_functions[function_name]
            tool_output = function_to_call(**function_args)
            
            print(f"خروجی تابع: {tool_output}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(tool_output)
            })
        
        # درخواست دوم (پاسخ نهایی)
        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )
        print("\n--- [Final Response (پاسخ نهایی مدل به کاربر)] ---")
        print(final_response.choices[0].message.content)
    else:
        print("\n--- [پاسخ مستقیم بدون ابزار] ---")
        print(response_message.content)

# اجرای تست برای بررسی فضای دیسک
run_agent("چقدر فضای آزاد توی مسیر / داریم؟")


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.